In [32]:
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append('../')
from src import *

In [33]:
X_train = joblib.load('../data/05/X_train_up.pkl')
X_valid = joblib.load('../data/05/X_valid_up.pkl')
X_test = joblib.load('../data/05/X_test_up.pkl')

y_train = joblib.load('../data/01/y_train.pkl')
y_valid = joblib.load('../data/01/y_valid.pkl')
y_test = joblib.load('../data/01/y_test.pkl')

In [34]:
lr_full = LogisticRegression(
    penalty="l2",
    C=1e6,
    solver="lbfgs",
    max_iter=1000
)

lr_full.fit(X_train, y_train)

coef = pd.Series(
    lr_full.coef_[0], 
    index=X_train.columns
).sort_values(key=abs, ascending=False)

best_gini = -1
best_n = None

for n in range(10, 40):
    top_features = coef.head(n).index

    lr_selected = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=5000
    )
    lr_selected.fit(X_train[top_features], y_train)

    lr_predict = lr_selected.predict_proba(X_valid[top_features])[:, 1]
    g = gini_score(y_valid, lr_predict)

    # print(f"TOP_N={n:2d} | GINI={g:.6f}")

    if g > best_gini:
        best_gini = g
        best_n = n

print(f"\nBest result: GINI={best_gini:.6f} with TOP_N={best_n}")



Best result: GINI=0.472920 with TOP_N=14


In [35]:
# 1. Обучаем Logistic Regression с L1 регуляризацией
lr_l1 = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=10.0,
    max_iter=5000
)

lr_l1.fit(X_train, y_train)

# 2. Смотрим какие признаки остались
coef_l1 = pd.Series(
    lr_l1.coef_[0],
    index=X_train.columns
)

selected_features_l1 = coef_l1[coef_l1 != 0].index.tolist()

print(f"Выбрано признаков L1: {len(selected_features_l1)}")

# 3. Строим урезанные датасеты
X_train_l1 = X_train[selected_features_l1]
X_valid_l1 = X_valid[selected_features_l1]

# 4. Обучаем обычную LR уже только на выбранных признаках
lr_after_l1 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=5000
)

lr_after_l1.fit(X_train_l1, y_train)

# 5. Считаем Gini
pred_valid_l1 = lr_after_l1.predict_proba(X_valid_l1)[:, 1]
gini_l1 = gini_score(y_valid, pred_valid_l1)

print(f"Gini после L1 feature selection: {gini_l1:.6f}")


Выбрано признаков L1: 39
Gini после L1 feature selection: 0.463741


In [36]:
joblib.dump(coef.head(14).index, '../data/06/selected_features_hand.pkl')
joblib.dump(selected_features_l1, '../data/06/selected_features_l1.pkl')

['../data/06/selected_features_l1.pkl']